Results show that the BFO<>IES mapping is consistent. The other two have a few problems. For CCOM<>QUDT, the parser has some problem with QUDT, and showed my mapping as inconsistent. An elimination test showed that QUDT is inconsistent on its own, so the mapping is not the issue. For CCOT<>TIME, W3C Time violates an OWL 2 DL restriction, so HermiT refused to reason over it. ELK reported success for all three because it passes with shallower evidence.

In [2]:

import shutil, subprocess, tempfile
from pathlib import Path

# --- Locate src/data from a notebook in notebooks/ (siblings) ---
NB_DIR   = Path.cwd().resolve()
SRC_DIR  = NB_DIR.parent / "src"
DATA_DIR = SRC_DIR / "data"

print("Notebook CWD:", NB_DIR)
print("SRC_DIR     :", SRC_DIR)
print("DATA_DIR    :", DATA_DIR)

if not SRC_DIR.exists() or not DATA_DIR.exists():
    raise FileNotFoundError("Could not find sibling src/ and src/data/ next to notebooks/.")

# --- Files to test (must live in src/data) ---
pairs = {
    "BFO–IES":   ["bfo-core.ttl", "ies.ttl",  "data/bfo-mapping.ttl"],
    "CCOM–QUDT": ["ccom.ttl",     "qudt.ttl", "data/ccom-mapping.ttl"],
    "CCOT–Time": ["ccot.ttl",     "time.ttl", "data/ccot-mapping.ttl"],
}

# --- Sanity: is ROBOT available? ---
if shutil.which("robot") is None:
    raise RuntimeError("ROBOT not found on PATH. Install ROBOT and make sure 'robot' is runnable.")

# Print robot version (optional; comment out if you prefer silence)
subprocess.run(["robot", "--version"], check=False)

reasoners = ["HermiT", "ELK"]

def run_robot_reason(files, reasoner):
    cmd = ["robot", "merge"]
    for f in files:
        cmd += ["--input", str(SRC_DIR / f)]
    with tempfile.NamedTemporaryFile(suffix=".ttl", delete=True) as tmp:
        cmd += ["reason", "--reasoner", reasoner, "--output", tmp.name]
        res = subprocess.run(cmd, capture_output=True, text=True)
    return res

for pair, files in pairs.items():
    for r in reasoners:
        res = run_robot_reason(files, r)
        status = "SUCCESS" if res.returncode == 0 else "FAILURE"
        print(f"{status}: {pair} [{r}]")
        if res.returncode != 0:
            print("   ", res.stderr.strip().splitlines()[-3:])

Notebook CWD: /Users/brianavann/repos/Ontology-Tradecraft/projects/project-3/assignment/notebooks
SRC_DIR     : /Users/brianavann/repos/Ontology-Tradecraft/projects/project-3/assignment/src
DATA_DIR    : /Users/brianavann/repos/Ontology-Tradecraft/projects/project-3/assignment/src/data


ROBOT version 1.9.11
SUCCESS: BFO–IES [HermiT]
SUCCESS: BFO–IES [ELK]
FAILURE: CCOM–QUDT [HermiT]
    ['WARNING: sun.misc.Unsafe::objectFieldOffset has been called by com.github.benmanes.caffeine.base.UnsafeAccess (file:/Users/brianavann/bin/robot.jar)', 'WARNING: Please consider reporting this to the maintainers of class com.github.benmanes.caffeine.base.UnsafeAccess', 'WARNING: sun.misc.Unsafe::objectFieldOffset will be removed in a future release']
SUCCESS: CCOM–QUDT [ELK]
FAILURE: CCOT–Time [HermiT]
    ['WARNING: sun.misc.Unsafe::objectFieldOffset has been called by com.github.benmanes.caffeine.base.UnsafeAccess (file:/Users/brianavann/bin/robot.jar)', 'WARNING: Please consider reporting this to the maintainers of class com.github.benmanes.caffeine.base.UnsafeAccess', 'WARNING: sun.misc.Unsafe::objectFieldOffset will be removed in a future release']
SUCCESS: CCOT–Time [ELK]


In [3]:
for pair, files in pairs.items():
    if pair == "BFO–IES":
        continue
    res = run_robot_reason(files, "HermiT")
    lines = [l for l in (res.stderr + res.stdout).splitlines() if not l.startswith("WARNING")]
    print("=====", pair)
    print("\n".join(lines[-15:]))

===== CCOM–QUDT
2026-11-09 21:12:56,707 ERROR org.semanticweb.owlapi.rdf.rdfxml.parser.OWLRDFConsumer - Entity not properly recognized, missing triples in input? http://org.semanticweb.owlapi/error#Error1 for type Class
2026-11-09 21:12:56,712 ERROR org.semanticweb.owlapi.rdf.rdfxml.parser.OWLRDFConsumer - Entity not properly recognized, missing triples in input? http://org.semanticweb.owlapi/error#Error2 for type Class
2026-11-09 21:12:56,713 ERROR org.semanticweb.owlapi.rdf.rdfxml.parser.OWLRDFConsumer - Entity not properly recognized, missing triples in input? http://org.semanticweb.owlapi/error#Error3 for type Class
2026-11-09 21:12:56,716 ERROR org.obolibrary.robot.IOHelper - Input ontology contains 6 triple(s) that could not be parsed:
 - _:genid-nodeid-node1k71tv55kx274 <http://www.w3.org/2002/07/owl#withRestrictions> _:genid-nodeid-node1k71tv55kx275.
 - _:genid-nodeid-node1k71tv55kx274 <http://www.w3.org/2002/07/owl#onDatatype> <http://www.w3.org/2001/XMLSchema#string>.
 - _:ge

In [4]:
tests = {
    "QUDT alone":           ["qudt.ttl"],
    "CCOM + QUDT":          ["ccom.ttl", "qudt.ttl"],
    "CCOM + QUDT + mapping":["ccom.ttl", "qudt.ttl", "data/ccom-mapping.ttl"],
}
for name, files in tests.items():
    res = run_robot_reason(files, "HermiT")
    inconsistent = "inconsistent" in (res.stderr + res.stdout)
    print(f"{name:25} returncode={res.returncode}  inconsistent={inconsistent}")

QUDT alone                returncode=1  inconsistent=True
CCOM + QUDT               returncode=1  inconsistent=True
CCOM + QUDT + mapping     returncode=1  inconsistent=True
